In [1]:
import os 
#os.environ["TAVILY_API_KEY"] = "***************************"
os.environ["TAVILY_API_KEY"] = "*******************************"

In [2]:
#!conda install fastmcp -y -vv

In [3]:
%%writefile server_multiple_tools.py
import os
from typing import Optional
from fastmcp import FastMCP
from tavily import TavilyClient

# Define validation constants
VALID_SEARCH_DEPTHS = {"advanced", "basic", "fast", "ultra-fast"}
VALID_TIME_RANGES = {"day", "week", "month", "year", "d", "w", "m", "y"}

def _execute_tavily_search(query: str, search_depth: str, max_results: int, topic: str, time_range: Optional[str], include_answer: bool, include_raw_content: bool, country: Optional[str], include_usage: bool) -> str:
    if not os.environ.get("TAVILY_API_KEY"):
        return "Validation Error: TAVILY_API_KEY environment variable is missing on the server process."
    errors = []
    # FIX: Clean up arguments passed from the LLM framework 
        # If any argument that should be a string/None is received as a dict, clean it up.
    cleaned_time_range = time_range
    if isinstance(time_range, dict):
        # If the dict has keys, grab the first value; otherwise default to None
        cleaned_time_range = list(time_range.values())[0] if time_range else "day"
    time_range = cleaned_time_range
    #if search_depth not in VALID_SEARCH_DEPTHS:
    #    errors.append(f"Invalid search_depth '{search_depth}' (Choose from: {VALID_SEARCH_DEPTHS})")
    #if not (1 <= max_results <= 20):
    #    errors.append(f"Invalid max_results '{max_results}' (Must be between 1 and 20)")
    #if time_range and time_range not in VALID_TIME_RANGES:
    #    errors.append(f"Invalid time_range '{time_range}' (Choose from: {VALID_TIME_RANGES})")
    #if errors:
    #    return "Validation Error: " + ", ".join(errors)
    try:
        tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))
        response = tavily_client.search(
            query=query, search_depth=search_depth, max_results=max_results,
            topic=topic, time_range=time_range, include_answer=include_answer,
            include_raw_content=include_raw_content, country=country, include_usage=include_usage
        )
        results = response.get("results", [])
        if not results:
            return f"Search completed successfully but returned no results for query: '{query}'."
        formatted_output = []
        if include_answer and response.get("answer"):
            formatted_output.append(f"Direct Answer: {response.get('answer')}\n" + "="*40)
        for idx, res in enumerate(results, start=1):
            formatted_output.append(f"[{idx}] Title: {res.get('title', 'No Title')}\nURL: {res.get('url', 'No URL')}\nSnippet: {res.get('content', 'No Content')}\n{'-'*40}")
        return "\n".join(formatted_output)
    except Exception as e:
        return f"Tavily API Runtime Exception: {str(e)}"

# A clean generator function ensures no automated loop parameters run on import
def get_initialized_server():
    mcp = FastMCP("Tavily Advanced Search Server")
    
    @mcp.tool()
    def search_general(query: str, search_depth: str = "basic", max_results: int = 5, time_range: Optional[str] = None, include_answer: bool = False, include_raw_content: bool = False, country: Optional[str] = None, include_usage: bool = False) -> str:
        """Execute a general search across the web for standard queries."""
        return _execute_tavily_search(query=query, search_depth=search_depth, max_results=max_results, topic="general", time_range=time_range, include_answer=include_answer, include_raw_content=include_raw_content, country=country, include_usage=include_usage)

    @mcp.tool()
    def search_news(query: str, search_depth: str = "basic", max_results: int = 5, time_range: Optional[str] = None, include_answer: bool = False, include_raw_content: bool = False, country: Optional[str] = None, include_usage: bool = False) -> str:
        """Execute a search targeted specifically at recent news articles and media."""
        return _execute_tavily_search(query=query, search_depth=search_depth, max_results=max_results, topic="news", time_range=time_range, include_answer=include_answer, include_raw_content=include_raw_content, country=country, include_usage=include_usage)

    @mcp.tool()
    def search_finance(query: str, search_depth: str = "basic", max_results: int = 5, time_range: Optional[str] = None, include_answer: bool = False, include_raw_content: bool = False, country: Optional[str] = None, include_usage: bool = False) -> str:
        """Execute a search targeted at financial markets, companies, and economic data."""
        return _execute_tavily_search(query=query, search_depth=search_depth, max_results=max_results, topic="finance", time_range=time_range, include_answer=include_answer, include_raw_content=include_raw_content, country=country, include_usage=include_usage)
    
    return mcp

if __name__ == "__main__":
    server_instance = get_initialized_server()
    server_instance.run()

Overwriting server_multiple_tools.py


In [4]:
#!conda install mcp -y -vv

In [5]:
from langchain_ollama import ChatOllama

# Step 2: Connect using the direct local Ollama channel
Model = ChatOllama(
    model="llama3.2:3b-instruct-q4_K_M",
    #model="gemma4:e4b",
    base_url="http://127.0.0.1:11434", # Notice: NO '/v1' path suffix needed here
    temperature=0.0,                   # Recommended baseline sampling for Gemma 4
    num_ctx=65536,                     # Opens the 16k context window for the framework
    num_predict=32768,                  # Provides enough room to write out markdown files
    num_batch=128  ,
    #format="json"
)

In [6]:
import sys
import os
import asyncio
import subprocess
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Patch event loop for Jupyter Notebook environments
nest_asyncio.apply()

current_python_exe = sys.executable
print(f"🔧 Targeting target environment: {current_python_exe}")

# Subprocess parameters configuration
server_params = StdioServerParameters(
    command=current_python_exe,
    args=["server_multiple_tools.py"],
    env={"TAVILY_API_KEY": os.environ.get("TAVILY_API_KEY")},
    stderr=subprocess.DEVNULL  
)

🔧 Targeting target environment: C:\ProgramData\anaconda3\envs\py__12_pytorch\python.exe


In [7]:
import os
from typing import Annotated, Sequence, TypedDict, Literal, Optional

print("[DEBUG 1] Starting clean imports...")
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
# IMPORT StructuredTool to bypass decorator version restrictions cleanly
from langchain_core.tools import StructuredTool 
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from fastmcp import Client

print("[DEBUG 2] Importing server function safely...")
from server_multiple_tools import get_initialized_server
server_instance = get_initialized_server()
print("[DEBUG 3] Server instance loaded into notebook shared memory space successfully!")

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

SYSTEM_INSTRUCTION = """
You are an expert Indian Share Market Analysis Agent. Your core task is to evaluate whether a user should BUY, HOLD, or SELL a specific Indian stock right now based on real-time data.

CRITICAL RULES:
1. You have multiple tools available. Choose the one that fits your need:
   - Use `search_finance` for stock prices, financials, corporate actions, and market data.
   - Use `search_news` for recent breaking news articles, press releases, and media stories.
   - Use `search_general` for high-level overview info or baseline standard queries.
2. Rely on fresh data. You MUST specify search_depth='advanced' to get comprehensive analytical reports.
3. If searching for recent updates, specify time_range='week' or time_range='month', default to time_range='day' if unsure.
4. Your final answer must clearly outline: Current Market Sentiments, Critical Technical / Fundamental Triggers, and a definitive action conclusion (BUY, HOLD, or SELL).

Important: When calling search tools, ensure that the country parameter is passed strictly as a plain text string. Do not structure it as an object or dictionary.
"""

def create_mcp_tool(name: str, docstring: str):
    """
    Constructs robust LangChain tools dynamically using StructuredTool 
    to guarantee explicit runtime configuration definitions.
    """
    async def _tool_func(
        query: str, 
        search_depth: Literal["basic", "advanced", "fast", "ultra-fast"] = "advanced", 
        max_results: int = 5,
        time_range: Optional[Literal["day", "week", "month", "year", "d", "w", "m", "y"]] = "week",
        include_answer: bool = False, 
        include_raw_content: bool = False,
        country: Optional[str] = None, 
        include_usage: bool = False
    ) -> str:
        print(f"\n[TOOL EXECUTION] '{name}' triggered with query: '{query}'")
        client = Client(server_instance)
        async with client:
            tool_result = await client.call_tool(
                name=name, 
                arguments={
                    "query": query, "search_depth": search_depth, "max_results": max_results,
                    "time_range": time_range, "include_answer": include_answer,
                    "include_raw_content": include_raw_content, "country": country, "include_usage": include_usage
                }
            )
            if hasattr(tool_result, "content") and isinstance(tool_result.content, list):
                return "\n".join([item.text for item in tool_result.content if hasattr(item, 'text')])
            return str(tool_result)
            
    # FIXED: Replaced explicit functional decorator calls with class constructors
    return StructuredTool.from_function(
        coroutine=_tool_func,
        name=name,
        description=docstring
    )

async def run_langgraph_mcp_agent(user_question: str, langchain_model):
    print("[DEBUG 4] Check: Is TAVILY_API_KEY set? ", "Yes" if os.environ.get("TAVILY_API_KEY") else "No")
    print("[DEBUG 5] Assembling tools...")
    
    tools_map = {
        "search_general": create_mcp_tool("search_general", "Execute a general search across the web for standard queries."),
        "search_news": create_mcp_tool("search_news", "Execute a search targeted specifically at recent news articles and media."),
        "search_finance": create_mcp_tool("search_finance", "Execute a search targeted at financial markets, companies, and economic data.")
    }
    model_with_tools = langchain_model.bind_tools(list(tools_map.values()))

    def call_model(state: AgentState):
        print("\n[NODE] Entering 'agent' node. Sending payload to LLM...")
        current_messages = state["messages"]
        if not any(isinstance(m, SystemMessage) for m in current_messages):
            current_messages = [SystemMessage(content=SYSTEM_INSTRUCTION)] + list(current_messages)
        response = model_with_tools.invoke(current_messages)
        print(f"[NODE] LLM responded. Tool calls requested: {response.tool_calls}")
        return {"messages": [response]}

    async def call_tools(state: AgentState):
        print("\n[NODE] Entering 'tools' node...")
        last_message = state["messages"][-1]
        tool_messages = []
        for tool_call in last_message.tool_calls:
            t_name = tool_call["name"]
            if t_name in tools_map:
                result_text = await tools_map[t_name].ainvoke(tool_call["args"])
                tool_messages.append(ToolMessage(content=result_text, tool_call_id=tool_call["id"]))
        return {"messages": tool_messages}

    def route_conditional(state: AgentState):
        has_calls = bool(state["messages"][-1].tool_calls)
        return "tools" if has_calls else END

    workflow = StateGraph(AgentState)
    workflow.add_node("agent", call_model)
    workflow.add_node("tools", call_tools)
    workflow.add_edge(START, "agent")
    workflow.add_conditional_edges("agent", route_conditional, {"tools": "tools", END: END})
    workflow.add_edge("tools", "agent")
    app = workflow.compile()

    print("[DEBUG 6] Graph workflow compiled completely. Starting stream...")
    inputs = {"messages": [HumanMessage(content=user_question)]}
    async for chunk in app.astream(inputs, stream_mode="values"):
        last_msg = chunk["messages"][-1]
        if last_msg.content:
            print(f"\n[STREAM] Update from {type(last_msg).__name__}:")
            print(str(last_msg.content)[:300] + "...")

print("\n[DEBUG 7] Ready. Invoking agent chain...")

[DEBUG 1] Starting clean imports...
[DEBUG 2] Importing server function safely...
[DEBUG 3] Server instance loaded into notebook shared memory space successfully!

[DEBUG 7] Ready. Invoking agent chain...


In [8]:
print('####################### Case 1 #############################')
await run_langgraph_mcp_agent("Should I buy or sell Reliance stock?", Model)


####################### Case 1 #############################
[DEBUG 4] Check: Is TAVILY_API_KEY set?  Yes
[DEBUG 5] Assembling tools...
[DEBUG 6] Graph workflow compiled completely. Starting stream...

[STREAM] Update from HumanMessage:
Should I buy or sell Reliance stock?...

[NODE] Entering 'agent' node. Sending payload to LLM...
[NODE] LLM responded. Tool calls requested: [{'name': 'search_finance', 'args': {'search_depth': 'advanced', 'include_answer': 'true', 'query': 'Reliance Industries stock price and analysis', 'country': 'IN'}, 'id': 'f8431971-2e16-4d9a-b54e-97b33b7b811f', 'type': 'tool_call'}, {'name': 'search_news', 'args': {'search_depth': 'advanced', 'time_range': 'day', 'include_answer': 'true', 'query': 'Reliance Industries news today', 'country': 'IN'}, 'id': 'fccaab44-5559-4646-bde7-39ae8b107957', 'type': 'tool_call'}]

[NODE] Entering 'tools' node...

[TOOL EXECUTION] 'search_finance' triggered with query: 'Reliance Industries stock price and analysis'

[TOOL EXECUTI

In [9]:
print('####################### Case 1 #############################')
await run_langgraph_mcp_agent("who is peter pan?", Model)


####################### Case 1 #############################
[DEBUG 4] Check: Is TAVILY_API_KEY set?  Yes
[DEBUG 5] Assembling tools...
[DEBUG 6] Graph workflow compiled completely. Starting stream...

[STREAM] Update from HumanMessage:
who is peter pan?...

[NODE] Entering 'agent' node. Sending payload to LLM...
[NODE] LLM responded. Tool calls requested: [{'name': 'search_general', 'args': {'query': 'peter pan', 'search_depth': 'advanced', 'country': 'en', 'include_answer': 'true', 'include_raw_content': 'false', 'include_usage': 'false', 'max_results': '10'}, 'id': '5f5c695b-75e8-4604-97dd-4c50bfbe9b4a', 'type': 'tool_call'}]

[NODE] Entering 'tools' node...

[TOOL EXECUTION] 'search_general' triggered with query: 'peter pan'

[STREAM] Update from ToolMessage:
Tavily API Runtime Exception: Invalid country. Must be a valid country name from the list of supported countries (https://docs.tavily.com/documentation/api-reference/endpoint/search)....

[NODE] Entering 'agent' node. Sending 

In [10]:
print('####################### Case 1 #############################')
await run_langgraph_mcp_agent("What is the lastest crude oil prices today?", Model)

####################### Case 1 #############################
[DEBUG 4] Check: Is TAVILY_API_KEY set?  Yes
[DEBUG 5] Assembling tools...
[DEBUG 6] Graph workflow compiled completely. Starting stream...

[STREAM] Update from HumanMessage:
What is the lastest crude oil prices today?...

[NODE] Entering 'agent' node. Sending payload to LLM...
[NODE] LLM responded. Tool calls requested: [{'name': 'search_finance', 'args': {'max_results': '10', 'search_depth': 'advanced', 'time_range': 'day', 'query': 'latest crude oil prices today', 'country': 'in', 'include_answer': 'true', 'include_raw_content': 'false', 'include_usage': 'false'}, 'id': '0ea600b8-8b01-4424-83c4-737715ef0b7b', 'type': 'tool_call'}]

[NODE] Entering 'tools' node...

[TOOL EXECUTION] 'search_finance' triggered with query: 'latest crude oil prices today'

[STREAM] Update from ToolMessage:
Tavily API Runtime Exception: Invalid country. Must be a valid country name from the list of supported countries (https://docs.tavily.com/d

In [11]:
async def run_agent():
    async with stdio_client(server_params, errlog=None) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            # Initialize connection
            await session.initialize()
            
            # List tools provided by your FastMCP script
            mcp_tools = await session.list_tools()
            
            # Format and list all tools clearly
            print("\n=== Available MCP Tools ===")
            for idx, tool in enumerate(mcp_tools.tools, start=1):
                print(f"\n[{idx}] Tool Name: {tool.name}")
                print(f"Description: {tool.description}")
                print("Arguments Schema:")
                # Prints out required parameters and their types
                properties = tool.inputSchema.get("properties", {})
                for param, details in properties.items():
                    print(f"  - {param} ({details.get('type')}): {details.get('description', 'No description')}")
            print("\n===========================")

if __name__ == "__main__":
    asyncio.run(run_agent())


=== Available MCP Tools ===

[1] Tool Name: search_general
Description: Execute a general search across the web for standard queries.
Arguments Schema:
  - query (string): No description
  - search_depth (string): No description
  - max_results (integer): No description
  - time_range (None): No description
  - include_answer (boolean): No description
  - include_raw_content (boolean): No description
  - country (None): No description
  - include_usage (boolean): No description

[2] Tool Name: search_news
Description: Execute a search targeted specifically at recent news articles and media.
Arguments Schema:
  - query (string): No description
  - search_depth (string): No description
  - max_results (integer): No description
  - time_range (None): No description
  - include_answer (boolean): No description
  - include_raw_content (boolean): No description
  - country (None): No description
  - include_usage (boolean): No description

[3] Tool Name: search_finance
Description: Execute 